# 09 Monitored Training

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`exp021_monitored_training.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp021_monitored_training.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)
- [`exp021_pixel_transform.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp021_pixel_transform.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)
- [`exp021_prepare.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp021_prepare.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)
- [`exp021_summarize_curves.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp021_summarize_curves.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/MLP/Normal/scripts/exp021_monitored_training.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'd272bdb3e82a8c7ce082ea9ffe21210131280bba'}, '05_model_optimize/MLP/Normal/scripts/exp021_pixel_transform.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'e55c6f30fd39e475963569456e6c2d9f978845b1'}, '05_model_optimize/MLP/Normal/scripts/exp021_prepare.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'cb26f036f2eb20363c04f2866143e7a7b8168a3f'}, '05_model_optimize/MLP/Normal/scripts/exp021_summarize_curves.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '9a2261d40f6818b749f3a5e1f473de1e064fd0e5'}}

## exp021_monitored_training

원본: [05_model_optimize/MLP/Normal/scripts/exp021_monitored_training.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp021_monitored_training.py)  
Git blob: `d272bdb3e82a8c7ce082ea9ffe21210131280bba`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

EXPO21: observed epoch-by-epoch OfficialVal loss/accuracy. OfficialVal is monitoring only (not used for fit/ES internal decision); no original/teacher Test.

CLI 인자 (정적 추출): `--seed`, `--early`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/exp021_monitored_training.py
#!/usr/bin/env python3
"""EXPO21: observed epoch-by-epoch OfficialVal loss/accuracy. OfficialVal is monitoring only (not used for fit/ES internal decision); no original/teacher Test."""
import os
for key in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'): os.environ[key]='1'
import argparse, pathlib, warnings, csv, json,time,sys
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
ROOT=pathlib.Path(__file__).resolve().parent
sys.path.insert(0,str(ROOT))
from pixel_transform import PixelScaler255

class MonitoredMLP(MLPClassifier):
 def _update_no_improvement_count(self, early_stopping, X, y, sample_weight):
  super()._update_no_improvement_count(early_stopping, X, y, sample_weight)
  i=len(self.loss_curve_)
  pp=self.predict_proba(self._monitor_x)
  p=np.clip(pp[np.arange(len(self._monitor_y)),self._monitor_y],1e-14,1.)
  val_nll=float(-np.mean(np.log(p)))
  val_acc=float(np.mean(pp.argmax(1)==self._monitor_y))
  internal=float(self.validation_scores_[-1]) if early_stopping else None
  info=dict(epoch=i,train_loss=float(self.loss_curve_[-1]),officialval_nll=val_nll,officialval_acc=val_acc,
   internal_val_acc=internal,loss_no_improvement=self._no_improvement_count)
  self._history.append(info)
  self._writer.writerow(info);self._logfile.flush()
  if i==1 or i%10==0:
   print('EPOCH',self.random_state,'ES',int(self.early_stopping),i,'train_loss',round(info['train_loss'],5),'val_nll',round(val_nll,5),'val_acc',round(val_acc,5),'internal',round(internal,5) if internal is not None else '-',flush=True)

if __name__=='__main__':
 p=argparse.ArgumentParser();p.add_argument('--seed',type=int,required=True,choices=[42,43,44]);p.add_argument('--early',action='store_true');a=p.parse_args()
 mode='ES' if a.early else 'NOES'
 xp=np.load(ROOT/'X_train_base.npy',mmap_mode='r')
 yt=np.load(ROOT/'y_train.npy')
 xv=np.load(ROOT/'X_val_com.npy',mmap_mode='r')
 yv=np.load(ROOT/'y_val.npy')
 X=np.asarray(xp,dtype=np.float32).copy()
 Xval=np.asarray(xv,dtype=np.float32).copy()/255.
 assert X.shape==(76800,784) and Xval.shape==(19200,784) and len(yv)==19200
 OUT=pathlib.Path('/tmp/exp021_curves');OUT.mkdir(exist_ok=True,parents=True)
 out=OUT/f'curve_{mode}_seed{a.seed}.csv';summary=OUT/f'fit_{mode}_seed{a.seed}.json'
 m=MonitoredMLP(hidden_layer_sizes=(256,128),alpha=.03,learning_rate_init=.001,activation='relu',solver='adam',max_iter=200,early_stopping=a.early,random_state=a.seed,
    n_iter_no_change=15 if a.early else 10, validation_fraction=.1)
 m._monitor_x=Xval;m._monitor_y=yv;m._history=[]
 with open(out,'w',encoding='utf-8',newline='') as f:
  m._logfile=f;m._writer=csv.DictWriter(f,fieldnames=['epoch','train_loss','officialval_nll','officialval_acc','internal_val_acc','loss_no_improvement']);m._writer.writeheader();f.flush()
  t=time.monotonic();print('START',mode,a.seed,flush=True)
  with warnings.catch_warnings(record=True) as notes:
   warnings.simplefilter('always');model=Pipeline([('scale',PixelScaler255()),('model',m)]);model.fit(X,yt)
  runtime=time.monotonic()-t
 final_pp=m.predict_proba(Xval)
 final_pred=np.asarray(final_pp.argmax(1))
 final_acc=float(np.mean(final_pred==yv))
 final_nll=float(-np.log(np.clip(final_pp[np.arange(len(yv)),yv],1e-14,1)).mean())
 final_7to9=int(((yv==7)&(final_pred==9)).sum())
 final_9to7=int(((yv==9)&(final_pred==7)).sum())
 del m._monitor_x,m._monitor_y,m._writer,m._logfile
 H=m._history
 ixacc=int(np.argmax([r['officialval_acc'] for r in H]));ixloss=int(np.argmin([r['officialval_nll'] for r in H]));
 rec=dict(mode=mode,seed=a.seed,final_restored_acc=final_acc,final_restored_nll=final_nll,final_restored_7to9=final_7to9,final_restored_9to7=final_9to7,n_iter=m.n_iter_,total_seconds=round(runtime,2),best_officialval_acc_epoch=ixacc+1,best_officialval_accuracy=H[ixacc]['officialval_acc'],best_officialval_loss_epoch=ixloss+1,best_officialval_nll=H[ixloss]['officialval_nll'],last_officialval_accuracy=H[-1]['officialval_acc'],last_officialval_nll=H[-1]['officialval_nll'],internal_best_epoch=int(np.argmax(m.validation_scores_))+1 if a.early else None,internal_best_score=m.best_validation_score_ if a.early else None,notes=[str(n.message) for n in notes],officialval_used_in_fit=False,original_test_used=False,professor_test_used=False,source='exact EXP017 80%size+14%stroke CoM; Official v3 76800/19200')
 summary.write_text(json.dumps(rec,ensure_ascii=False,indent=2));print('FINISH',json.dumps(rec),flush=True)


## exp021_pixel_transform

원본: [05_model_optimize/MLP/Normal/scripts/exp021_pixel_transform.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp021_pixel_transform.py)  
Git blob: `e55c6f30fd39e475963569456e6c2d9f978845b1`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

Exact upstream transformer: 03_experiments/W02_2026-11-09_11-15/scripts/experiment2_baseline/pixel_transform.py



In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/exp021_pixel_transform.py
"""Exact upstream transformer: 03_experiments/W02_2026-11-09_11-15/scripts/experiment2_baseline/pixel_transform.py"""
from sklearn.base import BaseEstimator, TransformerMixin
class PixelScaler255(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X / 255.0


## exp021_prepare

원본: [05_model_optimize/MLP/Normal/scripts/exp021_prepare.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp021_prepare.py)  
Git blob: `cb26f036f2eb20363c04f2866143e7a7b8168a3f`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/exp021_prepare.py
import os
for k in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS'):os.environ[k]='1'
from pathlib import Path
import json,sys,time
import numpy as np
root=Path('/mnt/data/EXP021_early_stopping_diagnostic')
sys.path.insert(0,str(root))
import train_local_warp as old
old.ROOT=root
old.V3=root/'MNIST_EMNIST_official_split_112000.npz'
t=time.monotonic();print('VERIFY_OFFICIAL',flush=True)
x,y,xval,yval=old.load_official()
train,report=old.build_base_train(x,y)
assert report['base_ink_mass']==2133060736.0,report
np.save(root/'X_train_base.npy',train);np.save(root/'y_train.npy',y)
np.save(root/'X_val_com.npy',old.com(xval));np.save(root/'y_val.npy',yval)
(root/'prep_manifest.json').write_text(json.dumps(report,indent=2))
print('PREP_DONE',round(time.monotonic()-t,2),report,flush=True)


## exp021_summarize_curves

원본: [05_model_optimize/MLP/Normal/scripts/exp021_summarize_curves.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp021_summarize_curves.py)  
Git blob: `9a2261d40f6818b749f3a5e1f473de1e064fd0e5`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

Aggregate a fully completed 3-seed EXP021 run. No new model fitting.

Monitoring external OfficialVal each epoch is diagnostic. Early stopping uses
sklearn's internal stratified 10% of pre-augmented OfficialTrain, not OfficialVal.



In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/exp021_summarize_curves.py
#!/usr/bin/env python3
"""Aggregate a fully completed 3-seed EXP021 run. No new model fitting.

Monitoring external OfficialVal each epoch is diagnostic. Early stopping uses
sklearn's internal stratified 10% of pre-augmented OfficialTrain, not OfficialVal.
"""
import os
for env in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS'):
    os.environ[env]='1'
import csv, json,sys,pathlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib

root=pathlib.Path('/mnt/data/EXP021_early_stopping_diagnostic')
tmp=pathlib.Path('/tmp/exp021_curves')
out=root/'final';out.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(root))
from pixel_transform import PixelScaler255

allrows=[];summ=[];source={}
for seed in (42,43,44):
    for mode in ('NOES','ES'):
        f=tmp/f'curve_{mode}_seed{seed}.csv'
        summfile=tmp/f'fit_{mode}_seed{seed}.json'
        if not f.exists() or not summfile.exists():
            raise FileNotFoundError(f'Incomplete run: {mode}/{seed}: {f}, {summfile}')
        df=pd.read_csv(f)
        rec=json.loads(summfile.read_text())
        assert len(df)==rec['n_iter'], (seed,mode,len(df),rec['n_iter'])
        assert df['epoch'].tolist()==list(range(1,len(df)+1))
        df.insert(0,'mode',mode);df.insert(1,'seed',seed)
        allrows.append(df)
        source[(seed,mode)]=(df,rec)
        best_loss=df.loc[df['officialval_nll'].idxmin()]
        best_acc=df.loc[df['officialval_acc'].idxmax()]
        if mode=='ES':
            restored_epoch=rec['internal_best_epoch']
            restored_acc=rec['final_restored_acc']
            restored_nll=rec['final_restored_nll']
            errors_7to9=rec['final_restored_7to9']
            errors_9to7=rec['final_restored_9to7']
        else:
            restored_epoch=rec['n_iter']
            restored_acc=rec.get('final_restored_acc',rec['last_officialval_accuracy'])
            restored_nll=rec.get('final_restored_nll',rec['last_officialval_nll'])
            errors_7to9=rec.get('final_restored_7to9',np.nan)
            errors_9to7=rec.get('final_restored_9to7',np.nan)
        summ.append(dict(seed=seed,mode=mode,train_rows=(69120 if mode=='ES' else 76800),internal_val_rows=(7680 if mode=='ES' else 0),actual_stopping_epoch=rec['n_iter'],restored_epoch=restored_epoch,reported_officialval_acc=restored_acc,reported_officialval_nll=restored_nll,raw_last_acc=rec['last_officialval_accuracy'],raw_last_nll=rec['last_officialval_nll'],best_officialval_acc_epoch=int(best_acc['epoch']),best_officialval_acc=float(best_acc['officialval_acc']),best_officialval_nll_epoch=int(best_loss['epoch']),best_officialval_nll=float(best_loss['officialval_nll']),internal_best_epoch=rec.get('internal_best_epoch'),internal_best_score=rec.get('internal_best_score'),err_7to9=errors_7to9,err_9to7=errors_9to7,training_time_seconds=rec['total_seconds']))

pd.concat(allrows,ignore_index=True).to_csv(out/'EXP021_epochwise_3seed_learning_curves.csv',index=False)
summary=pd.DataFrame(summ).sort_values(['seed','mode'])
summary.to_csv(out/'EXP021_comparison_by_seed.csv',index=False)

# Grounded comparison to the original frozen EXP017 model weights.
xval=np.load(root/'X_val_com.npy',mmap_mode='r')
yval=np.load(root/'y_val.npy')
verified=[]
for seed in (42,43,44):
    p=root/'frozen_models'/f'seed{seed}.joblib'
    m=joblib.load(p)
    probs=m.predict_proba(xval)
    pred=probs.argmax(1)
    verified.append(dict(seed=seed,officialval_acc=float(np.mean(pred==yval)),officialval_nll=float(-np.log(np.clip(probs[np.arange(len(yval)),yval],1e-14,1)).mean()),err_7to9=int(((yval==7)&(pred==9)).sum()),err_9to7=int(((yval==9)&(pred==7)).sum()),n_iter=m.named_steps['model'].n_iter_))
pd.DataFrame(verified).to_csv(out/'EXP021_original_frozen_model_check.csv',index=False)

fig,ax=plt.subplots(3,3,figsize=(17.4,11.7),sharex='col',constrained_layout=True)
for j,seed in enumerate((42,43,44)):
    for mode,label in (('NOES','No Early Stopping'),('ES','Early Stopping')):
        df,rec=source[(seed,mode)]
        ax[0,j].plot(df.epoch,df.train_loss,label=label,lw=1.6)
        ax[1,j].plot(df.epoch,df.officialval_nll,label=label,lw=1.6)
        ax[2,j].plot(df.epoch,100*df.officialval_acc,label=label,lw=1.6)
        if mode=='ES':
            ax[0,j].axvline(rec['internal_best_epoch'],linestyle='--',alpha=.5)
            ax[1,j].axvline(rec['internal_best_epoch'],linestyle='--',alpha=.5)
            ax[2,j].axvline(rec['internal_best_epoch'],linestyle='--',alpha=.5)
    ax[0,j].set_title(f'Seed {seed}',fontsize=13)
    ax[2,j].set_xlabel('Epoch')
    for i in range(3):ax[i,j].grid(alpha=.2)
ax[0,0].set_ylabel('Training loss\n(includes L2 term)')
ax[1,0].set_ylabel('OfficialVal log loss\n(no L2 term)')
ax[2,0].set_ylabel('OfficialVal accuracy (%)')
ax[0,2].legend(loc='upper right')
fig.suptitle('EXP021 | Epoch-by-epoch Learning Curves (alpha=0.03, 80%-Size + 14%-Stroke + CoM)\nDashed: ES internal best accuracy epoch, restored after stopping',fontsize=16)
fig.savefig(out/'EXP021_Learning_Curves_3Seeds.png',dpi=180,bbox_inches='tight')
plt.close(fig)

# Zoom to post-warmup plateaus, where stopping decisions matter.
fig,ax=plt.subplots(2,3,figsize=(17.4,8.2),constrained_layout=True)
for j,seed in enumerate((42,43,44)):
    for mode,label in (('NOES','Early Stopping OFF'),('ES','Early Stopping ON')):
        df,rec=source[(seed,mode)]
        cutoff=df[df.epoch>=10]
        ax[0,j].plot(cutoff.epoch,cutoff.officialval_nll,label=label)
        ax[1,j].plot(cutoff.epoch,100*cutoff.officialval_acc,label=label)
        if mode=='ES':
            ax[0,j].axvline(rec['internal_best_epoch'],linestyle='--',alpha=.5)
            ax[1,j].axvline(rec['internal_best_epoch'],linestyle='--',alpha=.5)
    noesdf,_=source[(seed,'NOES')]
    ax[0,j].axvline(int(noesdf.loc[noesdf.officialval_nll.idxmin(),'epoch']),linestyle=':',alpha=.55)
    ax[1,j].axvline(int(noesdf.loc[noesdf.officialval_acc.idxmax(),'epoch']),linestyle=':',alpha=.55)
    ax[0,j].set_title(f'Seed {seed}')
    ax[1,j].set_xlabel('Epoch (zoom: >=10)')
    for i in range(2):ax[i,j].grid(alpha=.22)
ax[0,0].set_ylabel('OfficialVal log loss')
ax[1,0].set_ylabel('OfficialVal accuracy (%)')
ax[1,-1].legend(fontsize=9)
fig.suptitle('EXP021 | Validation plateau and stopping point (post-warmup)',fontsize=15)
fig.savefig(out/'EXP021_Validation_Plateau_Zoom_3Seeds.png',dpi=180,bbox_inches='tight')
plt.close(fig)

fig,ax=plt.subplots(1,3,figsize=(16.5,4.8),constrained_layout=True,sharey=True)
for j,seed in enumerate((42,43,44)):
    df,rec=source[(seed,'ES')]
    ax[j].plot(df.epoch,100*df.internal_val_acc,label='Internal validation (used for ES)')
    ax[j].plot(df.epoch,100*df.officialval_acc,label='Official validation (diagnostic)')
    ax[j].axvline(rec['internal_best_epoch'],linestyle='--',alpha=.6)
    ax[j].set_title(f'Seed {seed}: restored epoch {rec["internal_best_epoch"]}')
    ax[j].set_xlabel('Epoch')
    ax[j].grid(alpha=.2)
ax[0].set_ylabel('Accuracy (%)')
ax[-1].legend(fontsize=9)
fig.suptitle('EXP021 | Early-stopping criterion vs external OfficialVal',fontsize=14)
fig.savefig(out/'EXP021_EarlyStopping_Internal_vs_Official_Accuracy.png',dpi=180,bbox_inches='tight')
plt.close(fig)

stats=summary.groupby('mode').agg(officialval_acc_mean=('reported_officialval_acc','mean'),officialval_acc_std=('reported_officialval_acc','std'),niter_mean=('actual_stopping_epoch','mean'),restored_epoch_mean=('restored_epoch','mean')).reset_index()
stats.to_csv(out/'EXP021_3Seed_aggregate.csv',index=False)
readme=[]
readme.append('# EXP021 — Early Stopping Learning Curves (실측 3-Seed)')
readme.append('')
readme.append('## 목적 및 모델 조건')
readme.append('')
readme.append('기존 EXP017 최종 MLP `(256,128)` / alpha=0.03 / learning rate=0.001 / Size80%-Stroke14%+CoM에 대해 Seed42/43/44에서 epoch별 Training Loss, 외부 OfficialVal Log Loss, 외부 OfficialVal Accuracy를 재측정하였다. 기존 frozen model weights는 변경하지 않았다.')
readme.append('')
readme.append('## 실험의 데이터 분할 차이')
readme.append('')
readme.append('- **NOES**: OfficialTrain 76,800 전체로 학습, `early_stopping=False`, `n_iter_no_change=10`, 최대 200 epoch, training-loss 기반 종료.')
readme.append('- **ES**: 동일 공식 Train에 대해 scikit-learn 내부에서 10%(7,680장)를 별도 검증으로 분리, 실제 가중치 학습 69,120장, `early_stopping=True`, `n_iter_no_change=15`, 최대 200 epoch, 내부 검증 **accuracy** 기반 중단 및 최고 score epoch 가중치 복원.')
readme.append('- 외부 OfficialVal 19,200장은 두 모델의 학습/내부 ES 판단에 사용되지 않았지만 **곡선 진단에 매 epoch 조회**하므로, external validation에서 후선정한 최고 epoch는 독립 Test 점수로 사용하거나 바로 배포하지 않는다.')
readme.append('- Original Test/교수님 Test/사용자 손글씨를 학습과 조기 종료 판단에 사용하지 않았다.')
readme.append('')
readme.append('## 관찰 결과')
readme.append('')
readme.append('| Seed | Mode | 실제 종료 | 복원 epoch | OfficialVal(최종) | OfficialVal loss 최저 epoch | OfficialVal accuracy 최고 epoch |')
readme.append('|---|---|---:|---:|---:|---:|---:|')
for x in summ:
    readme.append(f'| {x["seed"]} | {x["mode"]} | {x["actual_stopping_epoch"]} | {x["restored_epoch"]} | {x["reported_officialval_acc"]*100:.4f}% | {x["best_officialval_nll_epoch"]} | {x["best_officialval_acc_epoch"]} |')
readme.append('')
for row in stats.to_dict('records'):
    readme.append(f'- {row["mode"]}: 3 Seed 평균 OfficialVal 정확도 **{row["officialval_acc_mean"]*100:.4f}%**, 종료 epoch 평균 {row["niter_mean"]:.1f}, 복원 epoch 평균 {row["restored_epoch_mean"]:.1f}.')
readme.append('')
readme.append('**중요한 설계 한계:** ES와 NOES는 학습 데이터 수(69,120 vs 76,800)와 patience(15 vs 10)가 달라서, 성능 차이를 **Early Stopping의 독립 인과 효과**라고 단정할 수 없다. ES의 평가 상 이득이 있더라도 Train 분할 영향이 섞여 있다. 보다 엄격한 인과 비교에는 동일한 69,120 Train으로 같은 optimizer trajectory를 실행하여 체크포인트만 다른 epoch에서 선택해야 한다.')
readme.append('')
readme.append('## 실측 모델 선택에 필요한 추가 절차')
readme.append('')
readme.append('현재 ES가 내부 Validation Accuracy 기준으로 중단한 위치가 외부 Validation Loss/Accuracy에 비해 이르거나 늦었는지 곡선에서 확인하되, 조기 종료 정책 변경은 동일 분할/동일 학습 경로 비교와 크기 Shortcut Gate 검증을 마친 뒤에 결정한다.')
readme.append('')
readme.append('## 원본 파일')
readme.append('')
readme.append('- `EXP021_epochwise_3seed_learning_curves.csv`: 6개 실험의 매 epoch 수치')
readme.append('- `EXP021_comparison_by_seed.csv`, `EXP021_3Seed_aggregate.csv`: epoch 및 정확도 비교')
readme.append('- `EXP021_original_frozen_model_check.csv`: 원본 동결 모델과 버전 확인')
readme.append('- `EXP021_Learning_Curves_3Seeds.png`: 3×3 학습 곡선')
readme.append('- `EXP021_EarlyStopping_Internal_vs_Official_Accuracy.png`: 내부 검증 vs 외부 검증')
readme.append('- `EXP021_Validation_Plateau_Zoom_3Seeds.png`: 10 epoch 이후 검증 구간 확대')
(out/'EXP021_Study_Report.md').write_text('\n'.join(readme)+'\n',encoding='utf8')
print('OUTPUTS',*[p.name for p in out.iterdir()],sep='\n')
print(stats.to_string(index=False))


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")